# Cycle Skipping and Multiscale FWI

Understand nonconvex waveform objectives and compare frequency continuation with single-scale inversion.

## Computational conventions and data provenance
We solve the 2-D constant-density isotropic acoustic equation on a teaching grid. GeoFWI supplies the velocity values; the physical grid spacing and acquisition are assigned for this course, not field metadata. Numerical experiments share the same forward operator for observations and predictions (an inverse crime); they demonstrate principles rather than field-data performance. The sponge is approximate, and second-order spatial differences can be dispersive. Check points per shortest wavelength and reduce frequency or grid spacing for quantitative modeling.

**Data:** Li et al. (2026), *GeoFWI: A large velocity model dataset for benchmarking full waveform inversion using deep learning*, [paper](https://doi.org/10.1029/2025JH001037), [dataset](https://doi.org/10.5281/zenodo.18603709), [repository](https://github.com/aaspip/geofwi). The original archive contains 49,476 models of 100 × 100 samples. The official GeoFWI package downloads and caches the dataset; the selected model is then read from the local array. No synthetic substitute is silently used.

**Run in Colab:** Sign in, optionally select Runtime → Change runtime type → GPU, then execute cells in order. Each notebook is standalone. If download fails, rerun the data cell and check the network and available disk space. A fresh runtime loses its local cache. Save a copy in Drive to retain edits.


In [ ]:
# Colab normally includes these packages; install only missing dependencies.
import importlib.util, subprocess, sys
for package in ['numpy','scipy','matplotlib','torch','tqdm']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable,'-m','pip','install',package])
# Install the official package from its documented GitHub source.
if importlib.util.find_spec('geofwi') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','git+https://github.com/aaspip/geofwi.git'])
import numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter, zoom
from IPython.display import display
torch.manual_seed(7); np.random.seed(7)
torch.set_num_threads(2)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi':150,'savefig.dpi':300,'font.size':11,
    'axes.titlesize':12,'axes.titleweight':'semibold','axes.labelsize':11,
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':False,
    'lines.linewidth':2,'legend.frameon':False,'figure.facecolor':'white',
    'axes.facecolor':'white','image.interpolation':'nearest'})
print('Device:',device,'PyTorch:',torch.__version__)


## Runtime and data access
The [official GeoFWI Python interface](https://github.com/aaspip/geofwi) handles downloading, verification, extraction and caching. The first run requires a multi-gigabyte download and sufficient disk space; subsequent notebooks in the same runtime can reuse the cache. Separate Colab runtimes may need separate downloads. `find_arrays` discovers the files and `load` reads the velocity array. Select a model with its zero-based `MODEL_INDEX`; the loader does not assume a storage order or a fixed number of models. No synthetic fallback is used.

For geological categories, the official documentation also provides `geofwi.get_vel(layer=5, mode='salt', datapath=str(Path(path)/'geofwi'), indx=0)`. Here we use global indices and discover the actual file location rather than assuming an extraction directory.


In [ ]:
# Official GeoFWI API: download, discover, load and select.
import geofwi
MODEL_INDEX = 100  # zero-based index; change to explore other models
DATA_DIR = Path.home() / 'Datasets' / 'GeoFWI'
path = geofwi.download(path=str(DATA_DIR), progress=True)
print('GeoFWI dataset location:', path)
arrays = geofwi.find_arrays(path)
print('Available arrays:', arrays)
# Select by filename, not by the order in which files are discovered.
candidates = [p for p in arrays if Path(p).name == 'geofwi.npy']
if len(candidates) != 1:
    raise RuntimeError(f'Expected one geofwi.npy; found {candidates}. Available: {arrays}')
velocity = geofwi.load(candidates[0])
if velocity.ndim != 3 or velocity.shape[1:] != (100, 100):
    raise ValueError(f'Unexpected GeoFWI array shape: {velocity.shape}')
if not 0 <= MODEL_INDEX < velocity.shape[0]:
    raise IndexError(f'MODEL_INDEX must be between 0 and {velocity.shape[0]-1}')
v_native = np.array(velocity[MODEL_INDEX], dtype=np.float32, order='C', copy=True)
if not np.isfinite(v_native).all() or v_native.min() <= 0:
    raise ValueError('Invalid velocity model')
# GeoFWI velocities are in m/s.
if v_native.min() < 500 or v_native.max() > 10000:
    raise ValueError('Unexpected velocity range; check dataset units (m/s).')
N=48; dx=20.; dt=0.0015; nt=300
v_true_np=zoom(v_native,(N/100,N/100),order=1).astype('float32')
v_true=torch.tensor(v_true_np,device=device)
# Allow the same stable time step for the later velocity perturbations.
dt=min(dt, 0.6*dx/max(4500.,1.2*float(v_true.max())))
print(f'Dataset shape: {velocity.shape}; model index: {MODEL_INDEX}')
print(f'Velocity: {v_native.min():.1f}–{v_native.max():.1f} m/s; dt={dt:.6f} s')
fig, ax = plt.subplots(1,2,figsize=(10,4), layout='constrained')
lo,hi=float(v_native.min()),float(v_native.max())
for a,v,title in zip(ax,[v_native,v_true_np],['Original GeoFWI sample','Resampled teaching grid']):
    im=a.imshow(v,cmap='viridis',vmin=lo,vmax=hi,aspect='equal')
    a.set(title=title,xlabel='Horizontal sample',ylabel='Depth sample')
fig.colorbar(im,ax=ax,label='Velocity (m/s)',shrink=.85)
plt.show()


## Shared numerical engine
The following implementation is deliberately explicit so that every operation is inspectable. Source amplitude is in arbitrary units; the discrete forcing already includes the time-step scaling convention.

In [ ]:
# A differentiable second-order acoustic finite-difference solver.
# Array ordering: [shot, depth, horizontal]. All units are SI.
def ricker(f, t0=None):
    t=torch.arange(nt,device=device)*dt
    a=(torch.pi*f*(t-(1.5/f if t0 is None else t0)))**2
    return (1-2*a)*torch.exp(-a)
def forward(v, f=15., shots=(8,24,39), snapshots=False):
    assert float(v.detach().max())*dt/dx<1/np.sqrt(2), 'CFL violation'
    ns=len(shots); zsrc=5; zrec=5
    src=torch.zeros((ns,N,N),device=device)
    for i,x in enumerate(shots): src[i,zsrc,x]=1
    coords=torch.arange(N,device=device)
    # Sponge on all boundaries, not a free surface or PML.
    edge=torch.minimum(coords,N-1-coords).float()
    taper=torch.exp(-0.12*((5-edge).clamp(min=0)/5)**2)
    damp=taper[:,None]*taper[None,:]
    u=torch.zeros_like(src); old=torch.zeros_like(src)
    rec=[]; frames=[]; w=ricker(f); c=(v*dt/dx)**2
    for k in range(nt):
        p=torch.nn.functional.pad(u,(1,1,1,1))
        lap=p[:,2:,1:-1]+p[:,:-2,1:-1]+p[:,1:-1,2:]+p[:,1:-1,:-2]-4*u
        new=damp*(2*u-old+c*lap+src*w[k])
        old,u=u,new
        rec.append(u[:,zrec,5:-5])
        if snapshots and k in (60,120,180,240): frames.append(u[0].detach().cpu().numpy())
    result=torch.stack(rec,dim=1)
    return (result,frames) if snapshots else result
def arr(x): return x.detach().cpu().numpy() if torch.is_tensor(x) else np.asarray(x)
def modelplot(ax,v,title,limits=None):
    im=ax.imshow(arr(v),cmap='viridis',extent=[0,N*dx/1000,N*dx/1000,0],aspect='equal',**({} if limits is None else dict(vmin=limits[0],vmax=limits[1])))
    ax.set(xlabel='Distance (km)',ylabel='Depth (km)',title=title); plt.colorbar(im,ax=ax,label='m/s',shrink=.8)
def gather(ax,d,title,clip=None):
    a=arr(d); clip=np.percentile(np.abs(a),98) if clip is None else clip
    im=ax.imshow(a,aspect='auto',cmap='gray',vmin=-max(clip,1e-8),vmax=max(clip,1e-8),extent=[5*dx/1000,(N-6)*dx/1000,nt*dt,0]); ax.set(xlabel='Receiver x (km)',ylabel='Time (s)',title=title); plt.colorbar(im,ax=ax,shrink=.8,label='Amplitude (a.u.)')
def loss(pred,obs): return .5*(pred-obs).square().mean()/(obs.square().mean()+1e-12)
v0=torch.tensor(gaussian_filter(v_true_np,5),device=device)


## 1. Cycle skipping
For a sinusoid with time delay $\tau$, the period-averaged squared misfit is proportional to $1-\cos(2\pi f\tau)$. Local minima occur at full-period offsets. The half-period heuristic $|\Delta t|<1/(2f)$ describes the basin around zero in this idealized case, not a universal guarantee for broadband data. Low frequencies have wider basins.

In [ ]:
delay=np.linspace(-.2,.2,1000)
for f in [5,10,20]: plt.plot(delay,1-np.cos(2*np.pi*f*delay),label=f'{f} Hz')
plt.xlabel('Traveltime error (s)'); plt.ylabel('Normalized squared misfit'); plt.legend(); plt.show()

## 2. Frequency continuation
Compare 24 updates at 15 Hz with 8 updates each at 8, 11 and 15 Hz. This experiment regenerates observations with different source frequencies (an idealized frequency continuation); field FWI usually filters a fixed broadband survey. Adam settings, initial model and shot geometry are matched; the optimizer retains its state between stages.

In [ ]:
# A moderately biased starting model makes the problem harder.
initial=(v0*0.94).clamp(1300,4500)
def invert(initial, frequencies, steps=8, lr=.035):
    # Dimensionless parameter q=v/2000 improves optimizer scaling.
    q=torch.nn.Parameter(initial.detach().clone()/2000.)
    optimizer=torch.optim.Adam([q],lr=lr); history=[]; models=[]
    for f in frequencies:
        with torch.no_grad(): obs=forward(v_true,f)
        for i in range(steps):
            optimizer.zero_grad(); pred=forward(2000*q,f); objective=loss(pred,obs)
            objective.backward(); optimizer.step()
            with torch.no_grad(): q.clamp_(.65,2.25)
            history.append((f,float(objective.detach())))
        models.append((2000*q).detach().clone()); print(f'{f:g} Hz: {history[-1][1]:.4g}')
    return models[-1],np.array(history),models
single,hs,_=invert(initial,[15],steps=24)
multi,hm,stages=invert(initial,[8,11,15],steps=8)
limits=(float(v_true.min()),float(v_true.max()))
fig,ax=plt.subplots(2,2,figsize=(9,8))
for a,v,t in zip(ax.flat,[v_true,initial,single,multi],['True','Initial','Single scale','Multiscale']): modelplot(a,v,t,limits)
plt.tight_layout(); plt.show()
plt.semilogy(hs[:,1],label='15 Hz only'); plt.semilogy(hm[:,1],label='8 → 11 → 15 Hz'); plt.axvline(7.5,ls='--',c='gray'); plt.axvline(15.5,ls='--',c='gray'); plt.legend(); plt.xlabel('Iteration'); plt.ylabel('Stage-specific loss'); plt.show()

## 3. Fair comparison
Losses at different frequencies are not directly comparable. Evaluate all models at the common final frequency and using the same withheld acquisition. A single realization does not establish universal superiority of multiscale FWI.

In [ ]:
with torch.no_grad():
 obs=forward(v_true,15,shots=(16,32))
 for name,v in [('Initial',initial),('Single',single),('Multi',multi)]:
  print(name,'held-out loss',float(loss(forward(v,15,shots=(16,32)),obs)),'relative model error',float((v-v_true).norm()/v_true.norm()))
fig,ax=plt.subplots(1,3,figsize=(13,4))
for a,v,f in zip(ax,stages,[8,11,15]): modelplot(a,v,f'After {f} Hz',limits)
plt.tight_layout(); plt.show()

## 4. Exercises
Repeat with a stronger initial bias, different model indices, or an alternative frequency schedule. Explain the effects of missing low frequencies, finite recording time, and numerical dispersion. Compare final-frequency loss rather than concatenated stage loss alone.